# Gemma 3 270M 모델을 LiteRT (TFLite)로 변환하기

이 노트북은 Google의 Gemma 3 270M 모델을 LiteRT Edge Generative API를 사용하여
TFLite 모델로 변환하고 모바일 디바이스에서 실행 가능한 형태로 포팅하는 전체 과정을 보여줍니다.

## Gemma 3 270M 모델 특징

| 항목 | 값 |
|------|----|
| 파라미터 수 | ~270M |
| 레이어 수 | 18 |
| 임베딩 차원 | 640 |
| 어텐션 헤드 | 4 (head_dim=256) |
| GQA 그룹 | 1 (Grouped Query Attention) |
| FFN 크기 | 2048 (Gated, GELU_TANH) |
| Vocab 크기 | 262,144 |
| 최대 시퀀스 | 32,768 |
| 어텐션 패턴 | Global (매 6번째) + Local Sliding Window (512) |

## 워크플로우

1. 환경 설정
2. 체크포인트 다운로드 (Kaggle 또는 HuggingFace)
3. 모델 아키텍처 이해
4. 모델 빌드 및 가중치 로드
5. TFLite 변환 (multi-signature: prefill + decode)
6. 양자화
7. TFLite 추론 (텍스트 생성)
8. 검증 (원본 vs 변환 모델)

## 1. 환경 설정

In [ ]:
!pip install litert-torch
!pip install kagglehub  # Kaggle에서 체크포인트 다운로드용

In [ ]:
import os
import pathlib

import litert_torch
from litert_torch._convert import interface as converter_utils
from litert_torch.generative.examples.gemma3 import decoder
from litert_torch.generative.examples.gemma3 import gemma3
from litert_torch.generative.layers import kv_cache as kv_utils
import litert_torch.generative.layers.attention_utils as attn_utils
import litert_torch.generative.layers.model_config as cfg
from litert_torch.generative.quantize import quant_recipes
from litert_torch.generative.utilities import export_config as export_cfg
from litert_torch.generative.utilities import converter as conv_utils
from litert_torch.generative.utilities import model_builder
import litert_torch.generative.utilities.loader as loading_utils
import torch
from torch import nn

print(f"LiteRT Torch version: {litert_torch.__version__}")
print(f"PyTorch version: {torch.__version__}")

## 2. 체크포인트 다운로드

Gemma 3 270M 체크포인트는 Kaggle 또는 HuggingFace에서 다운로드할 수 있습니다.

### Kaggle (fused QKV 형식)
```python
import kagglehub
checkpoint = kagglehub.model_download("google/gemma-3/pyTorch/gemma-3-270m-it")
```

### HuggingFace (separate QKV 형식)
```bash
huggingface-cli download google/gemma-3-270m-it --local-dir ~/gemma-3-270m-it/
```

두 형식 모두 지원됩니다. 텐서 이름 매핑이 자동으로 처리됩니다.

In [ ]:
# 체크포인트 경로 설정
# 아래 중 하나를 선택하세요.

# 방법 1: Kaggle에서 다운로드
# import kagglehub
# CHECKPOINT_PATH = kagglehub.model_download("google/gemma-3/pyTorch/gemma-3-270m-it")

# 방법 2: 직접 경로 지정
CHECKPOINT_PATH = os.path.join(pathlib.Path.home(), "gemma-3-270m-it")

OUTPUT_DIR = "/tmp/gemma3_270m_tflite"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Checkpoint path: {CHECKPOINT_PATH}")
print(f"Checkpoint exists: {os.path.exists(CHECKPOINT_PATH)}")
print(f"Output dir: {OUTPUT_DIR}")

## 3. Gemma 3 270M 아키텍처 이해

Gemma 3 270M은 Decoder-only Transformer로, 다음과 같은 특징을 가집니다.

### 어텐션 패턴: Global + Local Sliding Window

매 6번째 레이어(idx 5, 11, 17)는 **Global Attention**을 사용하고,
나머지 레이어는 **Local Sliding Window Attention** (window_size=512)을 사용합니다.

```
Layer  0: Local (rotary_base=10,000)
Layer  1: Local
Layer  2: Local
Layer  3: Local
Layer  4: Local
Layer  5: Global (rotary_base=1,000,000)  ← 매 6번째
Layer  6: Local
...
Layer 11: Global
...
Layer 17: Global
```

### Decoder Block 구조

```
x ─► PreAttnNorm ─► Attention ─► PostAttnNorm ─► + (residual)
                                                  │
                                    ┌─────────────┘
                                    ▼
                              PreFFNorm ─► GatedFFN(GELU_TANH) ─► PostFFNorm ─► + (residual)
```

Gemma 3의 특징: `PostAttnNorm`을 residual 이전에 적용합니다 (일반 Transformer와 다름).

### LM Head
임베딩 가중치를 재사용합니다 (`lm_head.weight = tok_embedding.weight`).

In [ ]:
# 270M 모델 설정 확인
config_270m = decoder.get_decoder_config_270m()

print("=== Gemma 3 270M Config ===")
print(f"  vocab_size: {config_270m.vocab_size:,}")
print(f"  num_layers: {config_270m.num_layers}")
print(f"  embedding_dim: {config_270m.embedding_dim}")
print(f"  max_seq_len: {config_270m.max_seq_len:,}")
print(f"  embedding_scale: {config_270m.embedding_scale}")
print(f"  lm_head_use_bias: {config_270m.lm_head_use_bias}")
print()

print("=== Per-Layer Attention Config ===")
for i in range(config_270m.num_layers):
    bc = config_270m.block_config(i)
    ac = bc.attn_config
    attn_type = "GLOBAL" if ac.attn_type == cfg.AttentionType.GLOBAL else "LOCAL"
    print(
        f"  Layer {i:2d}: {attn_type:6s} | "
        f"heads={ac.num_heads}, head_dim={ac.head_dim}, "
        f"groups={ac.num_query_groups}, "
        f"rotary_base={ac.rotary_base:>10,}"
    )

print()
print("=== FFN Config ===")
ff = config_270m.block_config(0).ff_config
print(f"  type: {ff.type}")
print(f"  intermediate_size: {ff.intermediate_size}")
print(f"  activation: {ff.activation.type}")

## 4. 텐서 이름 매핑

Gemma 3는 두 가지 체크포인트 형식을 지원합니다:

### SafeTensors (HuggingFace) - Separate Q/K/V
```
model.layers.{i}.self_attn.q_proj  → attn_query_proj
model.layers.{i}.self_attn.k_proj  → attn_key_proj
model.layers.{i}.self_attn.v_proj  → attn_value_proj
```

### Kaggle Checkpoint - Fused QKV
```
model.layers.{i}.self_attn.qkv_proj  → attn_fused_qkv_proj
```

빌드 함수가 자동으로 두 형식을 모두 시도합니다.

In [ ]:
# 텐서 이름 매핑 확인
print("=== SafeTensors (HuggingFace) Tensor Names ===")
sep = decoder.TENSOR_NAMES_SEP_QKV
print(f"  embedding:        {sep.embedding}")
print(f"  attn_query_proj:  {sep.attn_query_proj}")
print(f"  attn_key_proj:    {sep.attn_key_proj}")
print(f"  attn_value_proj:  {sep.attn_value_proj}")
print(f"  attn_output_proj: {sep.attn_output_proj}")
print(f"  attn_query_norm:  {sep.attn_query_norm}")
print(f"  attn_key_norm:    {sep.attn_key_norm}")
print(f"  ff_gate_proj:     {sep.ff_gate_proj}")
print(f"  ff_up_proj:       {sep.ff_up_proj}")
print(f"  ff_down_proj:     {sep.ff_down_proj}")
print(f"  pre_attn_norm:    {sep.pre_attn_norm}")
print(f"  post_attn_norm:   {sep.post_attn_norm}")
print(f"  pre_ff_norm:      {sep.pre_ff_norm}")
print(f"  post_ff_norm:     {sep.post_ff_norm}")
print(f"  final_norm:       {sep.final_norm}")
print(f"  lm_head:          {sep.lm_head} (shared with embedding)")
print()

print("=== Kaggle Tensor Names ===")
fused = decoder.TENSOR_NAMES_FUSED_QKV
print(f"  embedding:          {fused.embedding}")
print(f"  attn_fused_qkv:     {fused.attn_fused_qkv_proj}")
print(f"  attn_query_norm:    {fused.attn_query_norm}")
print(f"  attn_key_norm:      {fused.attn_key_norm}")

## 5. 모델 빌드 및 가중치 로드

`gemma3.build_model_270m()`은 다음을 수행합니다:
1. `get_decoder_config_270m()`으로 모델 설정 생성
2. `Decoder` 클래스 인스턴스 생성
3. 체크포인트에서 가중치 로드 (safetensors/kaggle 형식 자동 감지)
4. `.eval()` 모드 설정

In [ ]:
# KV 캐시 설정
KV_CACHE_MAX_LEN = 1280  # prefill + decode 합산 최대 길이
PREFILL_SEQ_LENS = [8, 64, 128, 256, 512, 1024]  # 다양한 prefill 시그니처

print(f"KV cache max length: {KV_CACHE_MAX_LEN}")
print(f"Prefill sequence lengths: {PREFILL_SEQ_LENS}")

In [ ]:
# 모델 빌드
if os.path.exists(CHECKPOINT_PATH):
    gemma3_model = gemma3.build_model_270m(
        checkpoint_path=CHECKPOINT_PATH,
        mask_cache_size=KV_CACHE_MAX_LEN,  # 내부 마스크 캐시 사용 시
    )
    total_params = sum(p.numel() for p in gemma3_model.parameters())
    print(f"Model loaded successfully!")
    print(f"Total parameters: {total_params:,}")
    print(f"Model size (FP32): ~{total_params * 4 / 1024 / 1024:.0f} MB")
else:
    print(f"Checkpoint not found at: {CHECKPOINT_PATH}")
    print("Please download the checkpoint first (see Section 2).")

## 6. TFLite 변환

LLM은 두 가지 시그니처로 변환됩니다:

### `prefill_{seq_len}` 시그니처
- 입력 토큰 시퀀스 전체를 한 번에 처리
- 여러 시퀀스 길이(8, 64, 128, 256, 512, 1024)로 생성
- 추론 시 입력 길이에 가장 가까운 시그니처 선택

### `decode` 시그니처
- 토큰 1개씩 자기회귀적으로 생성
- KV 캐시를 활용하여 이전 계산 결과 재사용

### Gemma 3 특수 설정
- `mask_as_input=True`: 마스크를 외부 입력으로 전달 (Global+Local 어텐션 패턴 지원)
- `transpose_kv_cache=True`: KV 캐시 전치 형식 사용

In [ ]:
@torch.inference_mode()
def convert_gemma3_270m_to_tflite(
    model: nn.Module,
    output_dir: str,
    prefill_seq_lens: list = None,
    kv_cache_max_len: int = 1280,
    quantize: str = "dynamic_int8",
    mask_as_input: bool = True,
    transpose_kv_cache: bool = True,
) -> str:
    """Gemma 3 270M 모델을 multi-signature TFLite로 변환합니다.

    Args:
        model: 빌드된 Gemma 3 270M 모델
        output_dir: TFLite 출력 디렉토리
        prefill_seq_lens: prefill 시그니처에 사용할 시퀀스 길이 목록
        kv_cache_max_len: KV 캐시 최대 길이
        quantize: 양자화 방식 ('none', 'dynamic_int8', 'fp16', 등)
        mask_as_input: True면 마스크를 외부 입력으로 전달
        transpose_kv_cache: True면 전치된 KV 캐시 사용

    Returns:
        생성된 TFLite 파일 경로
    """
    if prefill_seq_lens is None:
        prefill_seq_lens = [8, 64, 128, 256, 512, 1024]

    config = decoder.get_decoder_config_270m()

    # Export 설정
    kv_layout = (
        kv_utils.KV_LAYOUT_TRANSPOSED
        if transpose_kv_cache
        else kv_utils.KV_LAYOUT_DEFAULT
    )
    export_config = export_cfg.ExportConfig(
        mask_as_input=mask_as_input,
        kvcache_layout=kv_layout,
    )

    # converter utility 사용
    output_file = conv_utils.convert_to_tflite(
        pytorch_model=model,
        output_path=output_dir,
        output_name_prefix="gemma3-270m",
        prefill_seq_len=prefill_seq_lens,
        kv_cache_max_len=kv_cache_max_len,
        quantize=quantize,
        config=config,
        export_config=export_config,
    )

    print(f"TFLite model saved to: {output_file}")
    size_mb = os.path.getsize(output_file) / (1024 * 1024)
    print(f"File size: {size_mb:.1f} MB")
    return output_file


print("Conversion function defined.")

In [ ]:
# 변환 실행 (Dynamic INT8 양자화)
if os.path.exists(CHECKPOINT_PATH):
    print("Starting conversion with dynamic INT8 quantization...")
    print(f"  Prefill lengths: {PREFILL_SEQ_LENS}")
    print(f"  KV cache max: {KV_CACHE_MAX_LEN}")
    print(f"  Mask as input: True")
    print(f"  Transpose KV cache: True")
    print()

    tflite_path = convert_gemma3_270m_to_tflite(
        model=gemma3_model,
        output_dir=OUTPUT_DIR,
        prefill_seq_lens=PREFILL_SEQ_LENS,
        kv_cache_max_len=KV_CACHE_MAX_LEN,
        quantize="dynamic_int8",
    )
else:
    print("Skipping conversion - checkpoint not available.")

## 7. 양자화 비교

다양한 양자화 옵션을 비교합니다.

| 옵션 | 가중치 | 연산 | 예상 크기 비율 | 특징 |
|------|--------|------|:----------:|------|
| `none` | FP32 | FP32 | 100% | 최고 정밀도 |
| `dynamic_int8` | INT8 | INT | ~25% | 기본 권장 |
| `weight_only_int8` | INT8 | FP | ~25% | 안정적 품질 |
| `fp16` | FP16 | FP | ~50% | 품질/크기 균형 |
| `dynamic_int4_block32` | INT4 | INT | ~15% | 극한 압축 (고품질) |
| `dynamic_int4_block128` | INT4 | INT | ~15% | 극한 압축 (고속) |

In [ ]:
# 다양한 양자화로 변환 (선택 사항)
QUANTIZE_OPTIONS = ["none", "dynamic_int8", "fp16"]

if os.path.exists(CHECKPOINT_PATH):
    results = {}
    for quant in QUANTIZE_OPTIONS:
        print(f"\n--- Converting with {quant} ---")
        path = convert_gemma3_270m_to_tflite(
            model=gemma3_model,
            output_dir=OUTPUT_DIR,
            prefill_seq_lens=[64, 256],  # 빠른 비교를 위해 줄임
            kv_cache_max_len=KV_CACHE_MAX_LEN,
            quantize=quant,
        )
        results[quant] = os.path.getsize(path) / (1024 * 1024)

    print("\n=== Quantization Size Comparison ===")
    base_size = results.get("none", 1)
    for quant, size_mb in results.items():
        ratio = size_mb / base_size * 100 if base_size else 0
        print(f"  {quant:25s}: {size_mb:7.1f} MB ({ratio:5.1f}%)")
else:
    print("Skipping quantization comparison - checkpoint not available.")

## 8. TFLite 모델로 텍스트 생성

변환된 TFLite 모델을 사용한 자기회귀 텍스트 생성 파이프라인입니다.

### 추론 흐름

```
1. Tokenize(prompt) → token_ids
2. Prefill: model(token_ids, input_pos, kv_cache, mask) → logits, kv_cache'
3. Sample: argmax(logits[-1]) → next_token
4. Decode loop:
   model(next_token, pos, kv_cache', mask) → logits, kv_cache''
   argmax(logits) → next_token
   반복...
5. Detokenize → 텍스트
```

In [ ]:
# PyTorch 모델로 직접 텍스트 생성 (TFLite 변환 전 검증용)

@torch.inference_mode()
def generate_with_pytorch(
    model: nn.Module,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 30,
    kv_cache_max_len: int = 1280,
) -> str:
    """PyTorch 모델로 텍스트를 생성합니다."""
    config = model.config

    # 토크나이즈
    chat_prompt = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
    input_ids = [2] + tokenizer.encode(chat_prompt)
    tokens = torch.tensor([input_ids], dtype=torch.int)
    seq_len = tokens.shape[1]

    # KV 캐시 초기화
    kv_cache = kv_utils.KVCache.from_model_config(
        kv_cache_max_len, config,
        kv_layout=kv_utils.KV_LAYOUT_TRANSPOSED,
    )

    # 마스크 캐시
    mask_cache = attn_utils.build_causal_mask_cache(kv_cache_max_len)

    # Prefill
    input_pos = torch.arange(0, seq_len, dtype=torch.int)
    mask = mask_cache.index_select(2, input_pos)
    output = model(tokens, input_pos, kv_cache, mask=mask)
    logits = output["logits"]
    kv_cache = output["kv_cache"]

    # 첫 번째 토큰 샘플링
    next_token = logits[0, -1].argmax().item()
    input_ids.append(next_token)

    # Decode 루프
    for i in range(max_new_tokens - 1):
        tokens = torch.tensor([[next_token]], dtype=torch.int)
        input_pos = torch.tensor([len(input_ids) - 1], dtype=torch.int)
        mask = mask_cache.index_select(2, input_pos)
        output = model(tokens, input_pos, kv_cache, mask=mask)
        logits = output["logits"]
        kv_cache = output["kv_cache"]

        next_token = logits[0, -1].argmax().item()
        input_ids.append(next_token)

        # EOS 체크 (Gemma EOS token)
        if next_token == 1:  # <eos>
            break

    return tokenizer.decode(input_ids)


print("Generation function defined.")
print("Note: Tokenizer는 별도로 로드해야 합니다 (sentencepiece).")

In [ ]:
# SentencePiece 토크나이저를 사용한 생성 예시
# Gemma 3는 SentencePiece 토크나이저를 사용합니다.

try:
    import sentencepiece as spm

    tokenizer_path = os.path.join(CHECKPOINT_PATH, "tokenizer.model")
    if os.path.exists(tokenizer_path) and os.path.exists(CHECKPOINT_PATH):
        sp = spm.SentencePieceProcessor()
        sp.Load(tokenizer_path)
        print(f"Tokenizer loaded: vocab_size={sp.GetPieceSize()}")

        # 생성 테스트
        prompt = "What is the meaning of life?"
        print(f"\nPrompt: {prompt}")
        result = generate_with_pytorch(
            gemma3_model, sp, prompt,
            max_new_tokens=30,
            kv_cache_max_len=KV_CACHE_MAX_LEN,
        )
        print(f"Output: {result}")
    else:
        print(f"Tokenizer not found at: {tokenizer_path}")
except ImportError:
    print("sentencepiece not installed. Run: pip install sentencepiece")

## 9. 변환 모델 검증

원본 모델과 재작성(reauthored) 모델의 출력을 비교하여 변환 정확도를 검증합니다.

In [ ]:
# 간단한 forward pass 검증

@torch.inference_mode()
def verify_model_output(model: nn.Module, kv_cache_max_len: int = 1280):
    """모델의 forward pass가 정상적으로 동작하는지 확인합니다."""
    config = model.config

    # 테스트 입력
    test_tokens = torch.tensor(
        [[2, 651, 9456, 576, 573, 3520, 3858, 603, 235248]], dtype=torch.int
    )
    seq_len = test_tokens.shape[1]
    input_pos = torch.arange(0, seq_len, dtype=torch.int)

    # KV 캐시
    kv_cache = kv_utils.KVCache.from_model_config(
        kv_cache_max_len, config,
        kv_layout=kv_utils.KV_LAYOUT_TRANSPOSED,
    )

    # 마스크
    mask_cache = attn_utils.build_causal_mask_cache(kv_cache_max_len)
    mask = mask_cache.index_select(2, input_pos)

    # Forward pass
    output = model(test_tokens, input_pos, kv_cache, mask=mask)

    logits = output["logits"]
    print(f"Output logits shape: {logits.shape}")
    print(f"Expected: (1, {seq_len}, {config.vocab_size})")

    # Top-5 예측 토큰
    top5 = torch.topk(logits[0, -1], 5)
    print(f"\nTop-5 predicted tokens:")
    for i, (val, idx) in enumerate(zip(top5.values, top5.indices)):
        print(f"  {i+1}. token_id={idx.item():6d}, logit={val.item():.4f}")

    print("\nForward pass verification: PASSED")
    return output


if os.path.exists(CHECKPOINT_PATH):
    verify_model_output(gemma3_model, KV_CACHE_MAX_LEN)
else:
    print("Skipping verification - checkpoint not available.")

In [ ]:
import numpy as np
from ai_edge_litert import interpreter as tflite_interp
import sentencepiece as spm


def generate_with_tflite(
    tflite_path: str,
    tokenizer_path: str,
    prompt: str,
    max_new_tokens: int = 30,
    kv_cache_max_len: int = 1280,
) -> str:
    """변환된 TFLite 모델로 텍스트를 생성합니다."""

    # 1. 토크나이저 로드
    sp = spm.SentencePieceProcessor()
    sp.Load(tokenizer_path)

    # 2. TFLite Interpreter 로드
    interp = tflite_interp.Interpreter(model_path=tflite_path)
    interp.allocate_tensors()
    sig_list = interp.get_signature_list()
    print(f"Available signatures: {list(sig_list.keys())}")

    # 3. 프롬프트 토크나이즈 (BOS + chat template 적용)
    chat_prompt = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
    token_ids = [2] + sp.encode(chat_prompt)  # 2 = <bos>
    print(f"Prompt tokens ({len(token_ids)}): {token_ids[:20]}...")

    # 4. 가장 적합한 prefill 시그니처 선택
    prefill_sigs = sorted(
        [k for k in sig_list if k.startswith("prefill_")],
        key=lambda k: int(k.split("_")[1]),
    )
    prefill_sig_name = None
    prefill_seq_len = None
    for sig_name in prefill_sigs:
        seq_len = int(sig_name.split("_")[1])
        if seq_len >= len(token_ids) - 1:  # -1: 마지막 토큰은 decode에서 사용
            prefill_sig_name = sig_name
            prefill_seq_len = seq_len
            break
    if prefill_sig_name is None:
        prefill_sig_name = prefill_sigs[-1]
        prefill_seq_len = int(prefill_sig_name.split("_")[1])
    print(f"Using signature: {prefill_sig_name} (seq_len={prefill_seq_len})")

    # 5. KV 캐시 초기화 (decode 시그니처에서 shape 추론)
    decode_runner = interp.get_signature_runner("decode")
    decode_input_details = sig_list["decode"]["inputs"]

    kv_cache = {}
    for input_name in decode_input_details:
        if input_name.startswith("kv_cache_"):
            # decode runner에서 input tensor의 shape/dtype 가져오기
            input_detail = decode_runner.get_input_details()[input_name]
            shape = input_detail["shape"]
            dtype = input_detail["dtype"]
            kv_cache[input_name] = np.zeros(shape, dtype=dtype)

    print(f"KV cache entries: {len(kv_cache)} (layers: {len(kv_cache) // 2})")
    if kv_cache:
        sample_key = list(kv_cache.keys())[0]
        print(f"  {sample_key} shape: {kv_cache[sample_key].shape}")

    # 6. 마스크 빌드 함수
    def build_causal_mask(seq_len, kv_max_len, start_pos=0):
        """Causal mask: [1, 1, seq_len, kv_max_len]"""
        mask = np.full((seq_len, kv_max_len), float("-inf"), dtype=np.float32)
        from ai_edge_litert.aot import aot_compile as aot_lib
from ai_edge_litert.aot.vendors.qualcomm import target as qnn_target

target_soc = qnn_target.Target(qnn_target.SocModel.SM8650)

compiled_models = aot_lib.aot_compile(
tflite_model_path="gemma_function_calling_int8.tflite",
targets=[target_soc]
)

qnn_model_path = "gemma_function_calling_qnn.tflite"
with open(qnn_model_path, "wb") as f:
f.write(compiled_models[target_soc]) for i in range(seq_len):
            mask[i, : start_pos + i + 1] = 0.0
        return mask.reshape(1, 1, seq_len, kv_max_len)

    # 7. Prefill 실행
    #    C++ 참조: 마지막 토큰 제외하고 prefill, 마지막 토큰부터 decode 시작
    effective_len = min(len(token_ids), prefill_seq_len + 1)
    prefill_len = effective_len - 1  # 마지막 토큰은 decode에서 사용

    prefill_tokens = np.zeros((1, prefill_seq_len), dtype=np.int32)
    prefill_input_pos = np.zeros(prefill_seq_len, dtype=np.int32)
    for i in range(prefill_len):
        prefill_tokens[0, i] = token_ids[i]
        prefill_input_pos[i] = i

    prefill_kwargs = {
        "tokens": prefill_tokens,
        "input_pos": prefill_input_pos,
        **kv_cache,
    }

    # mask_as_input=True로 변환된 경우 마스크 추가
    prefill_input_details = sig_list[prefill_sig_name]["inputs"]
    if "mask" in prefill_input_details:
        prefill_kwargs["mask"] = build_causal_mask(prefill_seq_len, kv_cache_max_len)

    prefill_runner = interp.get_signature_runner(prefill_sig_name)
    prefill_output = prefill_runner(**prefill_kwargs)

    # KV 캐시 업데이트 (prefill 출력에서)
    for key in kv_cache:
        if key in prefill_output:
            kv_cache[key] = prefill_output[key]

    # 8. Decode 루프
    next_token = token_ids[prefill_len]  # prefill에서 제외한 마지막 토큰
    next_pos = prefill_len
    output_tokens = []

    # EOS / end_of_turn 토큰
    eos_tokens = {1, sp.piece_to_id("<end_of_turn>")}

    for step in range(max_new_tokens):
        decode_tokens = np.array([[next_token]], dtype=np.int32)
        decode_input_pos = np.array([next_pos], dtype=np.int32)

        decode_kwargs = {
            "tokens": decode_tokens,
            "input_pos": decode_input_pos,
            **kv_cache,
        }
        if "mask" in sig_list["decode"]["inputs"]:
            decode_kwargs["mask"] = build_causal_mask(1, kv_cache_max_len, start_pos=next_pos)

        decode_runner = interp.get_signature_runner("decode")
        decode_output = decode_runner(**decode_kwargs)

        # KV 캐시 업데이트
        for key in kv_cache:
            if key in decode_output:
                kv_cache[key] = decode_output[key]

        # Greedy sampling (argmax)
        logits = decode_output["logits"]  # [1, 1, vocab_size]
        next_token = int(np.argmax(logits[0, -1, :]))
        output_tokens.append(next_token)
        next_pos += 1

        if next_token in eos_tokens:
            break

    # 9. 디토크나이즈
    output_text = sp.decode(output_tokens)
    return output_text


# === 실행 ===
if os.path.exists(CHECKPOINT_PATH):
    tokenizer_path = os.path.join(CHECKPOINT_PATH, "tokenizer.model")
    # tflite_path는 변환 셀에서 생성된 경로 사용
    # 예: /tmp/gemma3_270m_tflite/gemma3-270m_q8_ekv1280.tflite

    prompt = "What is the meaning of life?"
    print(f"Prompt: {prompt}\n")

    result = generate_with_tflite(
        tflite_path=tflite_path,  # 변환 셀에서 반환된 경로
        tokenizer_path=tokenizer_path,
        prompt=prompt,
        max_new_tokens=1000,
        kv_cache_max_len=KV_CACHE_MAX_LEN,
    )
    print(f"\nGenerated: {result}")

## 10. 변환 스크립트 (CLI) 사용법

노트북 대신 CLI 스크립트로도 변환할 수 있습니다.

### 기본 사용법
```bash
python litert_torch/generative/examples/gemma3/convert_gemma3_to_tflite.py \
    --model_size=270m \
    --checkpoint_path=~/gemma-3-270m-it/ \
    --output_path=/tmp/gemma3_tflite/ \
    --output_name_prefix=gemma3-270m \
    --quantize=dynamic_int8 \
    --prefill_seq_lens=8,64,128,256,512,1024 \
    --kv_cache_max_len=1280 \
    --mask_as_input=true \
    --transpose_kv_cache=true
```

### INT4 양자화
```bash
python litert_torch/generative/examples/gemma3/convert_gemma3_to_tflite.py \
    --model_size=270m \
    --checkpoint_path=~/gemma-3-270m-it/ \
    --quantize=dynamic_int4_block32
```

### GPU 백엔드 최적화
```bash
python litert_torch/generative/examples/gemma3/convert_gemma3_to_tflite.py \
    --model_size=270m \
    --checkpoint_path=~/gemma-3-270m-it/ \
    --gpu_dynamic_shapes=true
```

## 11. 모바일 배포

### 출력 파일 구조
변환된 TFLite 파일에는 다음 시그니처가 포함됩니다:

```
gemma3-270m_q8_ekv1280.tflite
├── prefill_8      (tokens[1,8], input_pos[8], kv_cache, mask)
├── prefill_64     (tokens[1,64], input_pos[64], kv_cache, mask)
├── prefill_128    (tokens[1,128], input_pos[128], kv_cache, mask)
├── prefill_256    (tokens[1,256], input_pos[256], kv_cache, mask)
├── prefill_512    (tokens[1,512], input_pos[512], kv_cache, mask)
├── prefill_1024   (tokens[1,1024], input_pos[1024], kv_cache, mask)
└── decode         (tokens[1,1], input_pos[1], kv_cache, mask)
```

### LiteRT Runtime으로 직접 호출
Android/iOS에서 LiteRT Runtime API로 시그니처를 선택하여 호출합니다.

### MediaPipe LLM Inference API
`prefill`과 `decode` 시그니처 네이밍 규칙을 따르므로,
SentencePiece 토크나이저와 함께 MediaPipe LLM Inference API로 배포할 수 있습니다.

### 모델 시각화
```bash
pip install ai-edge-model-explorer
model-explorer gemma3-270m_q8_ekv1280.tflite
```

## 12. SD 1.5와의 비교

| 항목 | Gemma 3 270M (LLM) | Stable Diffusion 1.5 |
|------|:------------------:|:--------------------:|
| **아키텍처** | Decoder-only Transformer | UNet + CLIP + VAE |
| **시그니처** | prefill_{N} + decode | encode + diffusion + decode |
| **KV 캐시** | 필요 (자기회귀 생성) | 불필요 |
| **마스크** | Causal + Sliding Window | Causal (CLIP만) |
| **TFLite 파일** | 1개 (multi-signature) | 3개 (컴포넌트별) |
| **양자화** | INT8/INT4 효과적 | weight-only INT8 권장 |
| **입력** | 텍스트 토큰 | 텍스트 + 노이즈 |
| **출력** | 텍스트 토큰 | 이미지 |
| **MediaPipe** | LLM Inference API 호환 | Runtime API 직접 사용 |

## 요약

이 노트북에서 다룬 내용:

1. Gemma 3 270M의 **아키텍처 분석** (Global + Local Sliding Window Attention)
2. **두 가지 체크포인트 형식** 지원 (HuggingFace safetensors / Kaggle fused QKV)
3. `model_builder.build_decoder_only_model()`로 **모델 빌드 및 가중치 로드**
4. `converter.convert_to_tflite()`로 **multi-signature TFLite 변환** (prefill + decode)
5. **6가지 양자화 옵션** 비교 (none, INT8, FP16, INT4 등)
6. KV 캐시 기반 **자기회귀 텍스트 생성** 파이프라인
7. **CLI 스크립트** 사용법 및 모바일 배포 가이드